# 51 — DeepEval Conversational Metrics
## Is the Dialogue Any Good? — Conversational Quality Metrics
⏱ ~12 min

Standard LLM evaluation scores **one turn at a time** — but production chatbots fail across turns. A bot may answer turn 1 correctly, forget the user's name by turn 3, and go off-topic by turn 5. Single-turn metrics catch none of this. **Conversational metrics** score the full trajectory.

By the end of this workshop you will know how to build `ConversationalTestCase` objects from real chat histories, run `KnowledgeRetentionMetric` and `ConversationRelevancyMetric`, compare stateful vs stateless chatbots quantitatively, and design conversations that stress-test each failure mode.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Single-turn vs multi-turn evaluation |
| 2 | **ConversationalTestCase** — Wrapping a turn history |
| 3 | **Stateful vs Stateless** — Running both chatbot types |
| 4 | **KnowledgeRetention** — Does the bot remember earlier facts? |
| 5 | **ConversationRelevancy** — Does the bot stay on-topic? |
| 6 | **Failure Modes** — Engineering conversations that break metrics |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the project `requirements.txt`
- An `OPENAI_API_KEY` in `.env` or Colab Secrets
- Familiarity with single-turn DeepEval (`LLMTestCase`, `AnswerRelevancyMetric`) — see Example 48

### Key References
> Xu, J., Szlam, A., & Weston, J. (2022). *Beyond Goldfish Memory: Long-Term Open-Domain Conversation.* ACL 2022. https://arxiv.org/abs/2107.07567
>
> Mehri, S., & Eskenazi, M. (2020). *DialoGLUE: A Natural Language Understanding Benchmark for Dialogue Systems.* https://arxiv.org/abs/2009.10855
>
> DeepEval Conversational Metrics docs: https://docs.confident-ai.com/docs/metrics-conversation-relevancy
>
> LangChain Chat Message History: https://python.langchain.com/docs/concepts/chat_history/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable,
            "-m", "pip", "install", "-q",
            "deepeval",
            "langchain",
            "langchain-openai",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os
import sys

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from deepeval.test_case import ConversationalTestCase, LLMTestCase, Turn
from deepeval.metrics import KnowledgeRetentionMetric, ConversationCompletenessMetric

# ── Sanity check ──────────────────────────────────────────────────────────────
key_ok = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY present: {key_ok}")
assert key_ok, (
    "\n  ACTION REQUIRED — add your key before running any LLM cells.\n"
    "  Local: echo 'OPENAI_API_KEY=sk-...' >> .env\n"
    "  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'"
)
print("Setup complete.")

---

## Part 1 — Single-Turn vs Multi-Turn Evaluation

---

### The Problem with Turn-by-Turn Scoring

A standard `LLMTestCase` measures **one question, one answer**. That works for a search box. It misses the dynamics that define conversational AI:

- Does the bot remember a fact the user shared two turns ago?
- Does it stay on-topic when the subject changes?
- Does it contradict itself between turns?

Research on production chatbot errors shows that **over 70% of failures are cross-turn** — the bot's per-turn quality looks fine, but the conversation as a whole breaks down.

---

### Metric Comparison

| Metric | Scope | What it measures | When to use |
|--------|-------|------------------|--------------|
| `AnswerRelevancyMetric` | Single turn | Is *this reply* relevant to *this question*? | FAQ bots, one-shot Q&A |
| `FaithfulnessMetric` | Single turn | Did *this reply* hallucinate beyond the context? | RAG pipelines |
| `KnowledgeRetentionMetric` | Whole conversation | Did the bot recall facts stated in earlier turns? | Any chatbot with memory |
| `ConversationCompletenessMetric` | Whole conversation | Did the bot address every user intent? | Task-oriented assistants |
| `ConversationCompletenessMetric` | Whole conversation | Did the bot address every user intent? | Task-oriented assistants |

---

### Multi-Turn Evaluation Flow

```
Turn 1: User: "My name is Alex."    → Bot: "Nice to meet you, Alex!"
Turn 2: User: "I work in DevOps."   → Bot: "DevOps is a great field."
Turn 3: User: "What's my name?"     → Bot: ???
                                              │
                              ┌───────────────┴──────────────────┐
                              │       ConversationalTestCase      │
                              │  turns = [turn1, turn2, turn3]   │
                              └───────────────┬──────────────────┘
                                              │
                              ┌───────────────┴──────────────────┐
                              │    KnowledgeRetentionMetric       │
                              │    looks back across all turns    │
                              │    → score: 1.0 (remembered)      │
                              │    → score: 0.0 (forgot)          │
                              └───────────────────────────────────┘
```

The key insight: the metric sees **all turns at once**, not just the last reply. It can detect whether the bot's answer to turn 3 is consistent with what the user said in turn 1.

In [ ]:
# ── 1-A: Building LLMTestCase and ConversationalTestCase ─────────────────────
#
# LLMTestCase wraps a single turn:
#   input         = the user's message
#   actual_output = the bot's reply
#
# ConversationalTestCase wraps the full trajectory:
#   turns = [Turn, Turn, ...]  — ordered oldest → newest
#
# Conversational metrics iterate over all turns looking for patterns
# that only emerge across the full history.

demo_turns = [
    LLMTestCase(
        input="My name is Alex and I work in DevOps.",
        actual_output="Nice to meet you, Alex! DevOps is a great field.",
    ),
    LLMTestCase(
        input="I'm currently migrating our infrastructure to Kubernetes.",
        actual_output="Kubernetes migrations can be complex — happy to help!",
    ),
    LLMTestCase(
        input="What's my name, and what infrastructure technology am I working with?",
        actual_output="Your name is Alex, and you're working with Kubernetes.",
    ),
]

def pair_cases_to_turns(pairs: list[LLMTestCase]) -> list[Turn]:
    """Convert legacy input/output pairs into current role-specific turns."""
    turns = []
    for pair in pairs:
        turns.extend([Turn(role="user", content=pair.input), Turn(role="assistant", content=pair.actual_output)])
    return turns

demo_case = ConversationalTestCase(turns=pair_cases_to_turns(demo_turns))

print(f"ConversationalTestCase has {len(demo_case.turns)} turns.")
print()
for i in range(0, len(demo_case.turns), 2):
    print(f"  Turn {i // 2 + 1}:")
    print(f"    User: {demo_case.turns[i].content}")
    print(f"    Bot:  {demo_case.turns[i + 1].content}")
    print()
print("Conversational eval scores the full trajectory — not just the last answer.")

---

## Part 2 — ConversationalTestCase: Wrapping a Turn History

---

### Common Failure Modes in Multi-Turn Conversations

| Failure mode | Description | Detected by |
|--------------|-------------|-------------|
| **Incomplete resolution** | Bot answers only part of a multi-intent request | `ConversationCompletenessMetric` |
| **Fact forgetting** | Bot fails to recall user-stated facts | `KnowledgeRetentionMetric` |
| **Self-contradiction** | Bot states X in turn 2, contradicts X in turn 5 | `KnowledgeRetentionMetric` |
| **Intent drop** | User asks three things; bot answers only one | `ConversationCompletenessMetric` |
| **Premature closure** | Bot declares task done before user confirms | `ConversationCompletenessMetric` |

### Stateful vs Stateless Chatbots

The architecture of the chatbot determines which failures are structurally possible:

| Architecture | Memory | Turn N sees turns 1..N-1? | Typical failures |
|--------------|--------|---------------------------|------------------|
| **Stateless** | None | No — each turn is independent | Fact forgetting, self-contradiction |
| **Stateful (full history)** | All messages | Yes | Context drift if history grows stale |
| **Stateful (windowed)** | Last k messages | Partial | Forgetting facts outside the window |

This workshop compares **fully stateful** (all messages in context) vs **stateless** (each turn sent with no history) to make the scoring difference visible.

In [ ]:
# ── 2-A: Helper — flat history dict list → current Turn objects ───────────────
#
# run_stateful_chat() and run_stateless_chat() both return a flat list:
#   [{"role": "user", "content": ...}, {"role": "assistant", "content": ...}, ...]
#
# Current DeepEval conversational metrics consume one role-specific Turn per message.


def history_to_turns(history: list[dict]) -> list[Turn]:
    """Convert flat role/content history to role-specific Turn objects."""
    return [Turn(role=message["role"], content=message["content"]) for message in history]


# Smoke-test the converter on a synthetic hand-crafted history
synthetic_history = [
    {"role": "user",      "content": "My name is Alex."},
    {"role": "assistant", "content": "Nice to meet you, Alex!"},
    {"role": "user",      "content": "What's my name?"},
    {"role": "assistant", "content": "Your name is Alex."},
]

converted = history_to_turns(synthetic_history)
print(f"Converted {len(synthetic_history)} messages → {len(converted)} Turn objects.")
for i, t in enumerate(converted):
    print(f"  Turn {i+1}: {t.role}='{t.content}'")

---

## Part 3 — Stateful vs Stateless: Running Both Chatbot Types

---

### How the Two Chatbots Differ

```
STATEFUL CHATBOT — full message history passed every turn
──────────────────────────────────────────────────────────
Turn 1:  [system] + [user: "My name is Alex"]              → LLM
Turn 2:  [system] + [user: "My name is Alex"]              \
                  + [bot: "Nice to meet you, Alex!"]         → LLM
                  + [user: "What am I working on?"]
Turn 3:  [system] + all previous turns + [user: ...]       → LLM

Result:  the model has full context every turn


STATELESS CHATBOT — fresh context every turn
──────────────────────────────────────────────────────────
Turn 1:  [system] + [user: "My name is Alex"]              → LLM
Turn 2:  [system] + [user: "What am I working on?"]        → LLM
Turn 3:  [system] + [user: "What's my name?"]              → LLM

Result:  the model has NO memory of previous turns
```

The `STATEFUL_CONVERSATION` test script is designed to expose this difference — it mentions facts in early turns and asks about them in later turns.

In [ ]:
# ── 3-A: Run both chatbots on the same conversation script ────────────────────
#
# The src/ modules provide:
#   STATEFUL_CONVERSATION  — a 5-turn script designed to test fact recall
#   RELEVANCE_CONVERSATION — a 4-turn script designed to test topic coherence
#   run_stateful_chat()    — sends full message history every turn
#   run_stateless_chat()   — sends only the current turn, no history

sys.path.insert(0, "..")  # make src importable when running from examples/51-*

from src.tools import STATEFUL_CONVERSATION, RELEVANCE_CONVERSATION
from src.workflow import run_stateful_chat, run_stateless_chat

print("Running stateful chatbot...")
stateful_history = run_stateful_chat(STATEFUL_CONVERSATION)
print(f"  Done. {len(stateful_history)} messages recorded.")

print("Running stateless chatbot...")
stateless_history = run_stateless_chat(STATEFUL_CONVERSATION)
print(f"  Done. {len(stateless_history)} messages recorded.")

print()
print("Conversation script:")
for role, content in STATEFUL_CONVERSATION:
    print(f"  [{role}] {content}")

In [ ]:
# ── 3-B: Side-by-side comparison on the fact-recall turns ─────────────────────
#
# Turn 3 (0-based index 2) is the first recall question: "What's my name?"
# The stateful bot should answer correctly; the stateless bot likely will not.

print("=" * 60)
print("TURN-BY-TURN COMPARISON")
print("=" * 60)

for turn_idx in range(len(STATEFUL_CONVERSATION)):
    msg_idx = turn_idx * 2  # flat list: user at even indices, bot at odd
    user_q   = stateful_history[msg_idx]["content"]
    sf_reply = stateful_history[msg_idx + 1]["content"]
    sl_reply = stateless_history[msg_idx + 1]["content"]

    print(f"\nTurn {turn_idx + 1}: {user_q}")
    print(f"  Stateful:  {sf_reply[:150]}")
    print(f"  Stateless: {sl_reply[:150]}")

print()
print("Note: on recall turns (3+), the stateless bot may fail or give generic answers.")

---

## Part 4 — KnowledgeRetentionMetric: Does the Bot Remember?

---

### How KnowledgeRetentionMetric Works

`KnowledgeRetentionMetric` asks an LLM judge to look at **every bot reply** and check whether it is consistent with facts established in **any earlier turn**.

```
Turn 1 (user):  "My name is Alex."
Turn 1 (bot):   "Nice to meet you, Alex!"
  → fact registered: name = Alex

Turn 3 (user):  "What's my name?"
Turn 3 (bot):   "I'm not sure — I don't have that information."
  → FAIL: contradicts established fact
  → score contribution: 0.0 for this turn

Final score: fraction of turns where established facts are honored
```

A score of **1.0** means every bot reply is consistent with the full conversation history. A score of **0.0** means the bot forgot or contradicted a fact on every applicable turn.

**Threshold:** 0.7 is a reasonable minimum for a production chatbot that handles user identity and preferences.

In [ ]:
# ── 4-A: KnowledgeRetentionMetric — stateful vs stateless ─────────────────────

stateful_case  = ConversationalTestCase(turns=history_to_turns(stateful_history))
stateless_case = ConversationalTestCase(turns=history_to_turns(stateless_history))

kr_metric = KnowledgeRetentionMetric(threshold=0.7, model="gpt-5.4-nano")

print("Running KnowledgeRetentionMetric...")
print()

for label, case in [("Stateful", stateful_case), ("Stateless", stateless_case)]:
    kr_metric.measure(case)
    status = "PASS" if kr_metric.is_successful() else "FAIL"
    bar = "█" * int(kr_metric.score * 20)
    print(f"{label}")
    print(f"  Score:  {kr_metric.score:.2f}  [{bar:<20}]  {status}")
    print(f"  Reason: {kr_metric.reason[:300]}")
    print()

print("Expected: stateful score > stateless score — the stateless bot has no memory.")

---

## Part 5 — ConversationCompletenessMetric: Did the Bot Address Every Intent?

---

### How ConversationCompletenessMetric Works

`ConversationCompletenessMetric` evaluates whether the assistant addressed all of the user's intents across the whole conversation.

It is sensitive to:
- Replies that wander off-topic mid-conversation
- Replies that answer a *previous* question instead of the current one
- Replies that are technically correct but miss the point of the user's question

```
RELEVANCE_CONVERSATION test script:

  Turn 1: "Tell me about Python programming."
  Turn 2: "What are the main data structures?"
  Turn 3: "How does garbage collection work in Python?"
  Turn 4: "What is the GIL?"

All turns are coherently Python-focused → high relevancy score.

Inject an off-topic reply at Turn 3:
  Bot: "I love making pasta! Here's a great carbonara recipe."

→ That turn gets a low relevancy score, pulling the overall score down.
```

In [ ]:
# ── 5-A: ConversationCompletenessMetric on RELEVANCE_CONVERSATION ─────────────

print("Running relevance conversation (stateful)...")
rel_history = run_stateful_chat(RELEVANCE_CONVERSATION)
rel_case = ConversationalTestCase(turns=history_to_turns(rel_history))

cr_metric = ConversationCompletenessMetric(threshold=0.7, model="gpt-5.4-nano")
cr_metric.measure(rel_case)

status = "PASS" if cr_metric.is_successful() else "FAIL"
bar = "█" * int(cr_metric.score * 20)
print(f"ConversationCompleteness")
print(f"  Score:  {cr_metric.score:.2f}  [{bar:<20}]  {status}")
print(f"  Reason: {cr_metric.reason[:300]}")
print()
print("The conversation stays on Python throughout — expect a high score.")

In [ ]:
# ── 5-B: Summary table — stateful vs stateless across both metrics ─────────────

print("Running all four metric combinations...")
print()

results = {}

for label, case in [("Stateful", stateful_case), ("Stateless", stateless_case)]:
    kr_metric.measure(case)
    kr_score = kr_metric.score

    cr_metric.measure(case)
    cr_score = cr_metric.score

    results[label] = {"KnowledgeRetention": kr_score, "ConversationCompleteness": cr_score}

print(f"{'Metric':<28} {'Stateful':>10} {'Stateless':>10} {'Delta':>8}")
print("-" * 60)

for metric_name in ["KnowledgeRetention", "ConversationCompleteness"]:
    sf    = results["Stateful"][metric_name]
    sl    = results["Stateless"][metric_name]
    delta = sf - sl
    delta_str = f"+{delta:.2f}" if delta >= 0 else f"{delta:.2f}"
    print(f"{metric_name:<28} {sf:>10.2f} {sl:>10.2f} {delta_str:>8}")

print()
print("A positive delta means stateful architecture outperforms stateless.")
print("KnowledgeRetention should show the largest gap — it requires cross-turn memory.")

---

## Part 6 — Failure Modes: Engineering for Score Drops

---

### Reading Metric Reasons

Both metrics return a `.reason` string alongside the score. This is the LLM judge's natural-language explanation of what it found. In production, **log these reasons** — they are far more actionable than a bare number.

### Common Patterns That Cause Score Drops

| Pattern | Score affected | Example |
|---------|---------------|---------|
| Bot says "I don't know your name" after user stated it | KnowledgeRetention ↓ | Turn 3 bot ignores turn 1 fact |
| Bot inserts a tangent between serious turns | ConversationRelevancy ↓ | "By the way, have you tried yoga?" |
| Bot contradicts itself across turns | KnowledgeRetention ↓ | Says Python is interpreted in T2, compiled in T4 |
| Bot re-asks a question the user already answered | ConversationRelevancy ↓ | "What is your name?" after user stated it |
| Bot completes only part of a compound request | ConversationCompleteness ↓ | User asks A + B, bot only answers A |

### Designing Adversarial Test Cases

The most useful evaluations are not the ones your bot passes — they are the ones designed to expose specific weaknesses:

1. **Memory stress test** — state a fact in turn 1, reference it in turns 3, 5, and 7
2. **Topic switch test** — pivot the conversation mid-way and check for drift
3. **Compound intent test** — pack three questions into one user message
4. **Contradiction test** — give the bot two contradictory facts and see which it honors

In [ ]:
# ── 6-A: Hard-coded adversarial test — score a known failure ─────────────────
#
# We build a ConversationalTestCase with a pre-written bot reply that
# deliberately forgets the user's name. No LLM call needed — we control
# actual_output directly. This is useful for unit-testing metric behavior.

forgetting_turns = [
    LLMTestCase(
        input="My name is Jordan and I'm a data scientist.",
        actual_output="Great to meet you, Jordan! Data science is a fascinating field.",
    ),
    LLMTestCase(
        input="I work mostly with Python and SQL.",
        actual_output="Python and SQL are a powerful combination for data work.",
    ),
    LLMTestCase(
        input="What's my name?",
        # Deliberately wrong — simulates a stateless or poorly-prompted bot
        actual_output="I'm sorry, I don't have access to your personal information.",
    ),
    LLMTestCase(
        input="And what do I do for work?",
        # Also wrong
        actual_output="I don't have information about your profession.",
    ),
]

forgetting_case = ConversationalTestCase(turns=pair_cases_to_turns(forgetting_turns))

kr_metric_adv = KnowledgeRetentionMetric(threshold=0.9, model="gpt-5.4-nano")
kr_metric_adv.measure(forgetting_case)

status = "PASS" if kr_metric_adv.is_successful() else "FAIL"
print("Adversarial test — bot that forgets immediately:")
print(f"  KnowledgeRetention score: {kr_metric_adv.score:.2f}  → {status}")
print(f"  Reason: {kr_metric_adv.reason[:400]}")
print()
print("Expected: the known forgetting score falls below the strict 0.9 production gate.")

In [ ]:
# ── 6-B: Hard-coded ideal case — score a perfect conversation ─────────────────
#
# A bot that consistently honors every fact gets a high retention score.
# Run this to confirm the metric rewards correct behavior.

ideal_turns = [
    LLMTestCase(
        input="My name is Sam. I live in Berlin and I'm a backend engineer.",
        actual_output="Nice to meet you, Sam! Berlin is a great city for tech. What kind of backend work do you do?",
    ),
    LLMTestCase(
        input="I mostly work with Go and Postgres.",
        actual_output="Go and Postgres are a solid combination — great performance characteristics.",
    ),
    LLMTestCase(
        input="What city am I in?",
        actual_output="You're in Berlin.",
    ),
    LLMTestCase(
        input="What programming language do I mainly use?",
        actual_output="You mainly use Go, along with Postgres for the database layer.",
    ),
    LLMTestCase(
        input="Can you summarize what you know about me?",
        actual_output="You're Sam, a backend engineer based in Berlin. You work primarily with Go and Postgres.",
    ),
]

ideal_case = ConversationalTestCase(turns=pair_cases_to_turns(ideal_turns))

kr_metric_ideal = KnowledgeRetentionMetric(threshold=0.7, model="gpt-5.4-nano")
kr_metric_ideal.measure(ideal_case)

status = "PASS" if kr_metric_ideal.is_successful() else "FAIL"
print("Ideal conversation — bot recalls every fact correctly:")
print(f"  KnowledgeRetention score: {kr_metric_ideal.score:.2f}  → {status}")
print(f"  Reason: {kr_metric_ideal.reason[:400]}")
print()
print("Expected: score close to 1.0 — every turn honors the established context.")

---

## Exercises

---

Work through these in order. Each builds on the patterns established in the workshop cells above. Starter templates are provided below each description.

**Exercise 1 — Inject an off-topic reply**
Take `RELEVANCE_CONVERSATION` and run it with the stateful chatbot. Then build a *hand-crafted* `ConversationalTestCase` where you replace turn 3's bot reply with a completely off-topic answer (e.g., the bot talks about cooking instead of Python garbage collection). Score both with `ConversationRelevancyMetric` and compare. How much does one bad turn drop the overall score?

**Exercise 2 — Design a perfect-retention conversation**
Build a 6-turn `ConversationalTestCase` with hard-coded outputs where the bot correctly recalls every fact the user shares across all turns. Verify that `KnowledgeRetentionMetric` returns ≥ 0.9. Try to find the *minimum* number of user-stated facts needed to make the metric's judgment interesting.

**Exercise 3 — Memory-critical task comparison**
Design a 5-turn task that *requires* memory to complete correctly (e.g., user sets a goal in turn 1, gives constraints in turn 3, asks for a recommendation in turn 5). Run both `run_stateful_chat` and `run_stateless_chat` on it. Score both with `KnowledgeRetentionMetric` and document the gap.

**Exercise 4 — Threshold sensitivity**
Take the `forgetting_case` from cell 6-A. Run `KnowledgeRetentionMetric` with thresholds 0.3, 0.5, 0.7, and 0.9. Print the PASS/FAIL status for each. What does this tell you about choosing evaluation thresholds for production use?

In [ ]:
# ── Exercise 1 Starter — off-topic injection ──────────────────────────────────

# Step 1: run the real conversation to get genuine bot replies
real_rel_history = run_stateful_chat(RELEVANCE_CONVERSATION)
real_rel_case = ConversationalTestCase(turns=history_to_turns(real_rel_history))

# Step 2: build a tampered version — replace turn 3's bot reply with off-topic content
tampered_turns = [
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[0][1],
        actual_output=real_rel_history[1]["content"],  # real reply
    ),
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[1][1],
        actual_output=real_rel_history[3]["content"],  # real reply
    ),
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[2][1],
        # TODO: replace this string with an off-topic reply (e.g. about cooking)
        actual_output="TODO: write an off-topic reply here",
    ),
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[3][1],
        actual_output=real_rel_history[7]["content"],  # real reply
    ),
]
tampered_case = ConversationalTestCase(turns=pair_cases_to_turns(tampered_turns))

# TODO: score both real_rel_case and tampered_case with ConversationCompletenessMetric
# TODO: print both scores and the drop
print("Fill in the off-topic reply above, then add scoring logic and re-run.")

In [ ]:
# ── Exercise 2 Starter — design a perfect-retention conversation ───────────────

my_ideal_turns = [
    LLMTestCase(
        input="TODO: user states at least two facts about themselves",
        actual_output="TODO: bot acknowledges and reflects those facts back",
    ),
    LLMTestCase(
        input="TODO: user adds more context",
        actual_output="TODO: bot responds naturally",
    ),
    LLMTestCase(
        input="TODO: user asks the bot to recall fact #1",
        actual_output="TODO: bot answers correctly",
    ),
    LLMTestCase(
        input="TODO: user asks about fact #2",
        actual_output="TODO: bot answers correctly",
    ),
    LLMTestCase(
        input="TODO: user asks about fact #3 (from turn 2)",
        actual_output="TODO: bot answers correctly",
    ),
    LLMTestCase(
        input="TODO: user asks for a complete summary",
        actual_output="TODO: bot gives a complete, accurate summary",
    ),
]

# TODO: create ConversationalTestCase and run KnowledgeRetentionMetric(threshold=0.9)
print("Fill in the turns above, then score with KnowledgeRetentionMetric(threshold=0.9).")

In [ ]:
# ── Exercise 3 Starter — memory-critical task comparison ──────────────────────

MEMORY_CRITICAL_CONVERSATION = [
    ("user", "TODO: turn 1 — user states a goal or constraint"),
    ("user", "TODO: turn 2 — user adds another piece of context"),
    ("user", "TODO: turn 3 — user adds a constraint that depends on turn 1"),
    ("user", "TODO: turn 4 — user asks a follow-up"),
    ("user", "TODO: turn 5 — user asks for a recommendation using facts from turns 1-3"),
]

# TODO: fill in the conversation, then run:
#   sf_history = run_stateful_chat(MEMORY_CRITICAL_CONVERSATION)
#   sl_history = run_stateless_chat(MEMORY_CRITICAL_CONVERSATION)
# TODO: score both with KnowledgeRetentionMetric and print a comparison table
print("Fill in the conversation turns above, then run both chatbots and score them.")

In [ ]:
# ── Exercise 4 Starter — threshold sensitivity ────────────────────────────────
# forgetting_case was built in cell 6-A above

thresholds = [0.3, 0.5, 0.7, 0.9]

print(f"{'Threshold':>10} {'Score':>8} {'Status':>8}")
print("-" * 30)

for t in thresholds:
    metric = KnowledgeRetentionMetric(threshold=t, model="gpt-5.4-nano")
    # TODO: call metric.measure(forgetting_case) and print the row
    print(f"{t:>10.1f} {'TODO':>8} {'TODO':>8}")

print()
print("Observation: a lenient threshold (0.3) passes cases a strict one (0.9) rejects.")

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

---

### Exercise 1 — Off-Topic Injection

**What to expect:** One off-topic turn should drop `ConversationRelevancyMetric` noticeably. With 4 turns total, one bad turn represents 25% of the conversation. Depending on how severely off-topic the injection is, expect the score to fall from ~0.9 to somewhere in the 0.5–0.7 range.

**Key insight:** Relevancy is averaged across turns. A single wildly irrelevant reply does not crater the score to zero — but it does pull it below the typical production threshold of 0.7, which is exactly the behavior you want from an evaluation system. This tells you that even one bad turn is detectable.

In [ ]:
# Sample answer — Exercise 1

real_rel_history_ans = run_stateful_chat(RELEVANCE_CONVERSATION)
real_rel_case_ans = ConversationalTestCase(turns=history_to_turns(real_rel_history_ans))

# Tampered: replace turn 3 bot reply with cooking content
tampered_turns_ans = [
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[0][1],
        actual_output=real_rel_history_ans[1]["content"],
    ),
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[1][1],
        actual_output=real_rel_history_ans[3]["content"],
    ),
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[2][1],
        actual_output=(
            "Speaking of things that take time to process — have you tried making "
            "sourdough bread? The fermentation process is fascinating and very "
            "rewarding. I highly recommend it as a weekend project!"
        ),
    ),
    LLMTestCase(
        input=RELEVANCE_CONVERSATION[3][1],
        actual_output=real_rel_history_ans[7]["content"],
    ),
]
tampered_case_ans = ConversationalTestCase(turns=pair_cases_to_turns(tampered_turns_ans))

cr_ans = ConversationCompletenessMetric(threshold=0.7, model="gpt-5.4-nano")

cr_ans.measure(real_rel_case_ans)
real_score = cr_ans.score

cr_ans.measure(tampered_case_ans)
tampered_score = cr_ans.score

print(f"Real conversation score:     {real_score:.2f}")
print(f"Tampered conversation score: {tampered_score:.2f}")
print(f"Score drop from one bad turn: {real_score - tampered_score:.2f}")

### Exercise 2 — Perfect-Retention Conversation

**What makes a perfect retention score:** Every time the bot's reply touches a fact the user stated earlier, it must do so accurately. The metric does not penalize turns where no recall is required — only turns where recall is possible but incorrect.

**Minimum viable stress test:** Two user-stated facts and two recall turns. With only two facts, the metric has clear signal. With six or more facts, scoring becomes harder to debug when something goes wrong.

In [ ]:
# Sample answer — Exercise 2

perfect_turns = [
    LLMTestCase(
        input="My name is Morgan. I'm a frontend engineer at a fintech startup.",
        actual_output="Nice to meet you, Morgan! Frontend at fintech sounds exciting — lots of complex UI challenges.",
    ),
    LLMTestCase(
        input="We're building a React dashboard that handles real-time stock data.",
        actual_output="A real-time stock dashboard in React — WebSockets and efficient state management will be key.",
    ),
    LLMTestCase(
        input="What's my name?",
        actual_output="Your name is Morgan.",
    ),
    LLMTestCase(
        input="What industry does my company work in?",
        actual_output="Your startup is in fintech — financial technology.",
    ),
    LLMTestCase(
        input="What framework are we using for the dashboard?",
        actual_output="You're using React for the dashboard.",
    ),
    LLMTestCase(
        input="Can you summarize everything you know about me and my project?",
        actual_output=(
            "You're Morgan, a frontend engineer at a fintech startup. "
            "You're building a real-time stock data dashboard using React."
        ),
    ),
]

perfect_case = ConversationalTestCase(turns=pair_cases_to_turns(perfect_turns))
kr_perfect = KnowledgeRetentionMetric(threshold=0.9, model="gpt-5.4-nano")
kr_perfect.measure(perfect_case)

status = "PASS" if kr_perfect.is_successful() else "FAIL"
print(f"Perfect retention score: {kr_perfect.score:.2f}  → {status}")
print(f"Reason: {kr_perfect.reason[:400]}")

### Exercise 3 — Memory-Critical Task Comparison

**Designing the task:** The most effective memory-critical tasks have a *dependency chain* — turn 5's answer only makes sense if the bot remembers turns 1 and 3. A simple "what's my name?" test is shallow; a recommendation that requires synthesizing multiple constraints is much harder.

**Expected gap:** On a well-designed memory-critical task, the stateful bot's `KnowledgeRetention` score should be 0.3–0.6 higher than the stateless bot. If the gap is small, the task may not actually require cross-turn memory — or the stateless bot is hallucinating plausible answers.

In [ ]:
# Sample answer — Exercise 3

TRAVEL_PLANNING_CONVERSATION = [
    ("user", "I want to plan a 5-day trip. My budget is $1500 total and I hate hot weather."),
    ("user", "I'm flying from New York."),
    ("user", "I also have a shellfish allergy, so I need good non-seafood food options."),
    ("user", "I prefer hiking and nature over city sightseeing."),
    ("user", "Given everything I've told you, what destination would you recommend and why?"),
]

print("Running stateful chatbot on travel planning task...")
sf_travel = run_stateful_chat(TRAVEL_PLANNING_CONVERSATION)
print("Running stateless chatbot on travel planning task...")
sl_travel = run_stateless_chat(TRAVEL_PLANNING_CONVERSATION)

sf_travel_case = ConversationalTestCase(turns=history_to_turns(sf_travel))
sl_travel_case = ConversationalTestCase(turns=history_to_turns(sl_travel))

kr_travel = KnowledgeRetentionMetric(threshold=0.7, model="gpt-5.4-nano")

kr_travel.measure(sf_travel_case)
sf_score = kr_travel.score

kr_travel.measure(sl_travel_case)
sl_score = kr_travel.score

print()
print(f"{'':20} {'Score':>8} {'Status':>8}")
print("-" * 38)
print(f"{'Stateful':20} {sf_score:>8.2f} {'PASS' if sf_score >= 0.7 else 'FAIL':>8}")
print(f"{'Stateless':20} {sl_score:>8.2f} {'PASS' if sl_score >= 0.7 else 'FAIL':>8}")
print(f"{'Gap':20} {sf_score - sl_score:>8.2f}")
print()
print("Stateless bot recommendation on turn 5:")
print(f"  {sl_travel[-1]['content'][:300]}")

### Exercise 4 — Threshold Sensitivity

**What to observe:** The same conversation and the same score can be PASS or FAIL depending entirely on the threshold. This is not a bug — it reflects that evaluation thresholds encode **your product's acceptable quality floor**, not a universal truth.

**Choosing thresholds in production:**
- Start lenient (0.5) while building your first eval dataset
- Tighten as you understand which scores correlate with user complaints
- Different use cases warrant different floors: a medical assistant needs 0.9+; a casual chatbot may tolerate 0.6

In [ ]:
# Sample answer — Exercise 4
# Re-build forgetting_case here for self-contained execution

forgetting_turns_ans = [
    LLMTestCase(
        input="My name is Jordan and I'm a data scientist.",
        actual_output="Great to meet you, Jordan! Data science is a fascinating field.",
    ),
    LLMTestCase(
        input="I work mostly with Python and SQL.",
        actual_output="Python and SQL are a powerful combination for data work.",
    ),
    LLMTestCase(
        input="What's my name?",
        actual_output="I'm sorry, I don't have access to your personal information.",
    ),
    LLMTestCase(
        input="And what do I do for work?",
        actual_output="I don't have information about your profession.",
    ),
]
forgetting_case_ans = ConversationalTestCase(turns=pair_cases_to_turns(forgetting_turns_ans))

# Measure once — the score is the same regardless of threshold
metric_ans = KnowledgeRetentionMetric(threshold=0.7, model="gpt-5.4-nano")
metric_ans.measure(forgetting_case_ans)
fixed_score = metric_ans.score

print(f"KnowledgeRetention score for the forgetting conversation: {fixed_score:.2f}")
print()
print(f"{'Threshold':>10} {'Score':>8} {'Status':>8}  Interpretation")
print("-" * 62)

notes = {
    0.3: "Very lenient — almost anything passes",
    0.5: "Lenient — allows significant forgetting",
    0.7: "Standard production threshold",
    0.9: "Strict — demands near-perfect retention",
}

for t in [0.3, 0.5, 0.7, 0.9]:
    status_str = "PASS" if fixed_score >= t else "FAIL"
    print(f"{t:>10.1f} {fixed_score:>8.2f} {status_str:>8}  {notes[t]}")

print()
print("Takeaway: threshold choice determines what you ship, not what the model does.")

---

## What's Next?

You now know how to evaluate the full trajectory of a multi-turn conversation. Here's where to go deeper:

### Measure more failure modes
- **Example 48 — DeepEval Hallucination & Bias** (`../48-deepeval-hallucination-bias/`): single-turn metrics for factual accuracy and demographic fairness — the foundation this workshop builds on. If you have not done it yet, start there.
- **Example 50 — DeepEval Agentic Eval** (`../50-deepeval-agentic-eval/`): score multi-step agent *tool-use trajectories*, not just conversation turns. The natural next step if your chatbot calls external tools.

### Generate test cases at scale
- **Example 52 — DeepEval Synthesizer** (`../52-deepeval-synthesizer/`): auto-generate hundreds of diverse conversational test cases from a knowledge base. Replaces hand-crafting the scenarios you built in these exercises — go here once you want coverage beyond five hand-written cases.

### Improve the chatbot under evaluation
- **Example 25 — Adaptive RAG** (`../25-adaptive-rag/`): plug retrieval into the chatbot so it can ground its answers in a knowledge base — directly improving `KnowledgeRetentionMetric` scores on factual conversations.
- **Example 30 — Agentic RAG** (`../30-agentic-rag/`): give the chatbot tools to look up information on demand, rather than relying entirely on its context window.

### Further reading
> Xu, J. et al. (2022). *Beyond Goldfish Memory: Long-Term Open-Domain Conversation.* ACL 2022. https://arxiv.org/abs/2107.07567
>
> DeepEval documentation — Conversational Metrics: https://docs.confident-ai.com/docs/metrics-conversation-relevancy
>
> LangChain Memory concepts: https://python.langchain.com/docs/concepts/chat_history/

---

*Workshop complete. The structural difference between stateful and stateless architectures is now quantified — use these metrics to hold your chatbot to account in CI.*